# Ablation Condition 2: Single Agent + Pre-Retrieved RAG/KG

A single LLM call per query that receives pre-retrieved context from all RAG sources (ChromaDB) and knowledge graphs.
The model must produce all three outputs (properties/constraints, candidate material, manufacturability) in one response.

Pre-retrieval uses the raw user query directly against all sources, without the multi-step query generation that MARS performs.

## Setup

In [1]:
import sys
import os
import json
import time
from pathlib import Path
from datetime import datetime

current_dir = Path().resolve()
if (current_dir / "src").exists() and (current_dir / "config").exists():
    project_root = current_dir
elif (current_dir.parent / "src").exists() and (current_dir.parent / "config").exists():
    project_root = current_dir.parent
else:
    project_root = Path(os.environ.get("SYS3DEV_ROOT", current_dir.parent))
sys.path.insert(0, str(project_root))
print(f"project_root: {project_root}")

from src.config import load_config, load_prompts
from src.utils import (
    llm,
    TransformerEmbeddingFunction,
    MaterialDatabase,
    PropertyMapper,
    load_ablation_queries,
    extract_json_from_response,
    build_ablation_evaluation,
    save_ablation_result,
    format_rag_results_for_prompt,
    format_kg_results_for_prompt,
)
from src.agents import ResearchAnalyst, ResearchScientist

from chromadb import PersistentClient
import networkx as nx
from sentence_transformers import SentenceTransformer
from GraphReasoning import load_embeddings

os.environ['TQDM_DISABLE'] = '1'
import tqdm
original_tqdm_init = tqdm.tqdm.__init__
def disabled_tqdm_init(self, *args, **kwargs):
    kwargs['disable'] = True
    return original_tqdm_init(self, *args, **kwargs)
tqdm.tqdm.__init__ = disabled_tqdm_init

config = load_config()
prompts = load_prompts()
print("Configuration and prompts loaded")

project_root: /orcd/pool/004/tphage/SG_march/MARS_ablation3
Configuration and prompts loaded


## Initialize LLM and Embeddings

In [2]:
llm_config = {
    "api_key": config["llm"]["api_key"],
    "base_url": config["llm"]["base_url"],
    "model": config["llm"]["model_name"],
    "max_tokens": config["llm"]["max_tokens"],
}
llm_instance = llm(llm_config)
generate = llm_instance.generate_cli
temperature = config["llm"].get("temperature", 0)
print(f"LLM initialized: {config['llm']['model_name']}")

embedding_tokenizer = ''
embedding_model = SentenceTransformer(config["embeddings"]["model_name"], trust_remote_code=True)
embedding_function = TransformerEmbeddingFunction(
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
)
print("Embedding model initialized")

LLM initialized: gpt-oss-20b


<All keys matched successfully>


Embedding model initialized


## Load Knowledge Graphs

In [3]:
graphs_cfg = config["data"]["graphs"]
kg_dir = graphs_cfg["kg_dir"]

# Material Properties KG
mp_cfg = graphs_cfg["material_properties"]
G_materialproperties = nx.read_graphml(os.path.join(kg_dir, mp_cfg["graph_file"]))
relation = nx.get_edge_attributes(G_materialproperties, "title")
nx.set_edge_attributes(G_materialproperties, relation, "relation")
nx.set_node_attributes(G_materialproperties, nx.pagerank(G_materialproperties), "pr")
node_embeddings_mp = load_embeddings(os.path.join(kg_dir, mp_cfg["embedding_file"]))
print(f"Material Properties KG loaded: {G_materialproperties}")

# PFAS KG
pfas_kg_cfg = graphs_cfg["pfas"]
G_pfas = nx.read_graphml(os.path.join(kg_dir, pfas_kg_cfg["graph_file"]))
relation_pfas = nx.get_edge_attributes(G_pfas, "title")
nx.set_edge_attributes(G_pfas, relation_pfas, "relation")
nx.set_node_attributes(G_pfas, nx.pagerank(G_pfas), "pr")
node_embeddings_pfas = load_embeddings(os.path.join(kg_dir, pfas_kg_cfg["embedding_file"]))
print(f"PFAS KG loaded: {G_pfas}")

# Patent KG
patent_cfg = graphs_cfg["patents"]
G_patents = nx.read_graphml(os.path.join(kg_dir, patent_cfg["graph_file"]))
relation_patents = nx.get_edge_attributes(G_patents, "title")
nx.set_edge_attributes(G_patents, relation_patents, "relation")
nx.set_node_attributes(G_patents, nx.pagerank(G_patents), "pr")
node_embeddings_patents = load_embeddings(os.path.join(kg_dir, patent_cfg["embedding_file"]))
print(f"Patent KG loaded: {G_patents}")
print("All knowledge graphs loaded")

Material Properties KG loaded: DiGraph with 317800 nodes and 774009 edges
PFAS KG loaded: DiGraph with 144335 nodes and 459248 edges
Patent KG loaded: DiGraph with 199008 nodes and 578361 edges
All knowledge graphs loaded


## Load ChromaDB Collections

In [4]:
chroma_cfg = config["data"]["chromadb"]
base_path = chroma_cfg.get("base_path", "")
n_results = 5

rag_analysts = {}

# PFAS
pfas_chroma_cfg = chroma_cfg["pfas"]
pfas_db_path = os.path.join(base_path, pfas_chroma_cfg["database_path"]) if base_path else pfas_chroma_cfg["database_path"]
client_pfas = PersistentClient(path=pfas_db_path)
pfas_collection = client_pfas.get_collection(pfas_chroma_cfg["collection_name"], embedding_function=embedding_function)
rag_analysts["pfas_papers"] = ResearchAnalyst(collection=pfas_collection, embedding_function=embedding_function, n_results=n_results)
print("PFAS ChromaDB loaded")

# Patents
patents_chroma_cfg = chroma_cfg["patents"]
patents_db_path = os.path.join(base_path, patents_chroma_cfg["database_path"]) if base_path else patents_chroma_cfg["database_path"]
client_patents = PersistentClient(path=patents_db_path)
patents_collections = client_patents.list_collections()
patents_coll_name = patents_chroma_cfg["collection_name"] or patents_collections[0].name
patents_collection = client_patents.get_collection(patents_coll_name, embedding_function=embedding_function)
rag_analysts["patents"] = ResearchAnalyst(collection=patents_collection, embedding_function=embedding_function, n_results=n_results)
print("Patents ChromaDB loaded")

# MaterialDB
matdb_cfg = chroma_cfg["materialdb"]
matdb_db_path = os.path.join(base_path, matdb_cfg["database_path"]) if base_path else matdb_cfg["database_path"]
client_matdb = PersistentClient(path=matdb_db_path)
matdb_collections = client_matdb.list_collections()
matdb_coll_name = matdb_cfg["collection_name"] or matdb_collections[0].name
matdb_collection = client_matdb.get_collection(matdb_coll_name, embedding_function=embedding_function)
rag_analysts["materialdb"] = ResearchAnalyst(collection=matdb_collection, embedding_function=embedding_function, n_results=n_results)
print("MaterialDB ChromaDB loaded")

# Manufacturing Textbooks
mfg_cfg = chroma_cfg.get("manufacturing_textbooks", {})
if mfg_cfg:
    try:
        mfg_db_path = os.path.join(base_path, mfg_cfg["database_path"]) if base_path else mfg_cfg["database_path"]
        if os.path.exists(mfg_db_path):
            client_mfg = PersistentClient(path=mfg_db_path)
            mfg_collections = client_mfg.list_collections()
            mfg_coll_name = mfg_cfg.get("collection_name") or (mfg_collections[0].name if mfg_collections else None)
            if mfg_coll_name:
                mfg_collection = client_mfg.get_collection(mfg_coll_name, embedding_function=embedding_function)
                rag_analysts["manufacturing_textbooks"] = ResearchAnalyst(collection=mfg_collection, embedding_function=embedding_function, n_results=n_results)
                print("Manufacturing Textbooks ChromaDB loaded")
    except Exception as e:
        print(f"Skipping manufacturing_textbooks: {e}")

print(f"RAG sources available: {list(rag_analysts.keys())}")

PFAS ChromaDB loaded
Patents ChromaDB loaded
MaterialDB ChromaDB loaded
Manufacturing Textbooks ChromaDB loaded
RAG sources available: ['pfas_papers', 'patents', 'materialdb', 'manufacturing_textbooks']


## Initialize KG Scientists and Material Database

In [5]:
scientist_mp = ResearchScientist(
    knowledge_graph=G_materialproperties,
    node_embeddings=node_embeddings_mp,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    algorithm="shortest",
)

scientist_pfas = ResearchScientist(
    knowledge_graph=G_pfas,
    node_embeddings=node_embeddings_pfas,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    algorithm="shortest",
)

scientist_patents = ResearchScientist(
    knowledge_graph=G_patents,
    node_embeddings=node_embeddings_patents,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    algorithm="shortest",
)

print("KG scientists initialized")

# Material database
property_mapper = PropertyMapper(embedding_model=embedding_model, embedding_tokenizer=embedding_tokenizer)
material_db_path = config.get("data", {}).get("material_database", {}).get("path", "")
material_db = MaterialDatabase.load_from_json(material_db_path, property_mapper=property_mapper)
print(f"Material database loaded: {len(material_db)} materials")

KG scientists initialized
Material database loaded: 20 materials


## Load Queries and Prompts

In [6]:
queries = load_ablation_queries()
ablation_prompts = prompts["ablation"]

print(f"Loaded {len(queries)} benchmark queries:")
for q in queries:
    print(f"  - {q['name']}: {q['sentence'][:80]}...")

Loaded 6 benchmark queries:
  - Query1: Find a substitute material for PFAS material THV (elastomeric terpolymer of tetr...
  - Query2: Identify PFAS free thermoplastic material for heat shrink tubing with a 2:1 shri...
  - Query3: Identify non fluoropolymer thermoplastics that can withstand severe thermal shoc...
  - Query4: What non-PFAS additives or formulation strategies can be used in thermoplastics ...
  - Query5: Beyond PEEK, which thermoplastic materials can:
- Withstand continuous or interm...
  - PTFE_Seals: Find a substitute material for PTFE that can be used in industrial seals applica...


## Define Pre-Retrieval and Pipeline

In [7]:
def pre_retrieve_context(query, rag_analysts, scientists, material_db):
    """Pre-retrieve RAG and KG context using the raw query."""
    context_parts = []
    sentence = query["sentence"]
    keywords = [query["material_X"], query["application_Y"]]

    # RAG retrieval from all sources
    for source_name, analyst in rag_analysts.items():
        try:
            result = analyst.analyze_question(sentence)
            rag_results = result.get("rag_results", [])
            context_parts.append(format_rag_results_for_prompt(rag_results, source_name))
            print(f"    RAG [{source_name}]: {len(rag_results)} documents")
        except Exception as e:
            print(f"    RAG [{source_name}]: Error - {e}")
            context_parts.append(f"[{source_name}]: Retrieval failed.\n")

    # KG retrieval
    for kg_name, scientist in scientists.items():
        try:
            kg_result = scientist.find_connections(keywords)
            context_parts.append(format_kg_results_for_prompt(kg_result, kg_name))
            n_nodes = len(kg_result.get("matched_node_ids", []))
            n_paths = len(kg_result.get("found_paths", []))
            print(f"    KG [{kg_name}]: {n_nodes} matched nodes, {n_paths} paths")
        except Exception as e:
            print(f"    KG [{kg_name}]: Error - {e}")
            context_parts.append(f"[{kg_name} KG]: Connection search failed.\n")

    # Material database summary
    mat_lines = ["--- Available Materials Database ---"]
    for mat in material_db.materials:
        mat_name = mat.get("material_name", mat.get("material_id", "unknown"))
        mat_class = mat.get("material_class", "")
        mat_lines.append(f"- {mat_name} ({mat_class})")
    context_parts.append("\n".join(mat_lines))

    return "\n\n".join(context_parts)


def run_1agent_rag_ablation(query, generate_fn, ablation_prompts, rag_analysts, scientists, material_db, temperature=0):
    """Run the single-agent + RAG/KG ablation for a single query."""
    raw_responses = {}
    start_time = time.time()

    # Pre-retrieve context
    print("  Pre-retrieving context...")
    retrieved_context = pre_retrieve_context(query, rag_analysts, scientists, material_db)
    retrieval_time = time.time() - start_time
    print(f"  Pre-retrieval complete ({retrieval_time:.1f}s, {len(retrieved_context)} chars)")

    # Single LLM call
    print("  Running single-agent LLM call...")
    system_prompt = ablation_prompts["single_agent_with_context"]
    user_prompt = ablation_prompts["single_agent_with_context_user_prompt"].format(
        sentence=query["sentence"],
        material_X=query["material_X"],
        application_Y=query["application_Y"],
        retrieved_context=retrieved_context,
    )
    response_raw = generate_fn(system_prompt=system_prompt, prompt=user_prompt, temperature=temperature)
    raw_responses["single_agent"] = response_raw
    raw_responses["retrieved_context_chars"] = len(retrieved_context)

    parsed = extract_json_from_response(response_raw)
    if parsed is None:
        print("    WARNING: Failed to parse response as JSON")
        parsed = {}

    # Extract fields
    rmp = parsed.get("required_material_properties", {})
    properties = rmp.get("properties", [])
    constraints = rmp.get("constraints", [])

    cs = parsed.get("candidate_selection", {})
    candidate = cs.get("final_candidate", cs) if cs else None

    manufacturing = parsed.get("manufacturing_process", {
        "status": "unknown",
        "process_recipe": None,
        "blocking_constraints": [],
        "feedback_to_system2": "",
    })

    print(f"    Properties: {len(properties)}, Constraints: {len(constraints)}")
    if candidate:
        print(f"    Candidate: {candidate.get('material_name', 'N/A')}")
    print(f"    Mfg status: {manufacturing.get('status', 'unknown')}")

    duration = time.time() - start_time

    return build_ablation_evaluation(
        query=query,
        properties=properties,
        constraints=constraints,
        candidate=candidate,
        manufacturing=manufacturing,
        condition_name="1agent_rag",
        run_id=datetime.now().strftime("%Y%m%d%H"),
        duration_seconds=duration,
        raw_responses=raw_responses,
    )

## Run All Queries

In [8]:
kg_scientists = {
    "material_properties": scientist_mp,
    "pfas": scientist_pfas,
    "patents": scientist_patents,
}

results = {}
output_base = os.path.join(project_root, "pipeline_logs")

for i, query in enumerate(queries, 1):
    name = query["name"]
    print(f"\n{'='*70}")
    print(f"[{i}/{len(queries)}] Running 1-Agent+RAG/KG ablation for: {name}")
    print(f"{'='*70}")
    print(f"  Query: {query['sentence']}...")

    result = run_1agent_rag_ablation(
        query, generate, ablation_prompts,
        rag_analysts, kg_scientists, material_db,
        temperature=temperature,
    )
    results[name] = result

    output_dir = os.path.join(output_base + f"_{name}")
    path = save_ablation_result(result, output_dir, "1agent_rag", result["metadata"]["pipeline_run_id"])
    print(f"  Saved to: {path}")
    print(f"  Duration: {result['metadata']['duration_seconds']:.1f}s")

print(f"\n{'='*70}")
print("All 1-Agent+RAG/KG ablation runs complete.")
print(f"{'='*70}")


[1/6] Running 1-Agent+RAG/KG ablation for: Query1
  Query: Find a substitute material for PFAS material THV (elastomeric terpolymer of tetrafluoroethylene, hexafluoropropylene, and vinylidene fluoride) that has broad chemical resistance, including acids, bases, protic and aprotic solvents, oils, and salt solutions. Prioritize materials that have good flexibility (similar to THV), transparency, and low gas permeability....
  Pre-retrieving context...
    RAG [pfas_papers]: 5 documents
    RAG [patents]: 5 documents
    RAG [materialdb]: 5 documents
    RAG [manufacturing_textbooks]: 5 documents
    KG [material_properties]: 2 matched nodes, 1 paths
    KG [pfas]: 2 matched nodes, 1 paths
    KG [patents]: 2 matched nodes, 1 paths
  Pre-retrieval complete (150.7s, 40529 chars)
  Running single-agent LLM call...
    Properties: 7, Constraints: 4
    Candidate: DIC.PPS (cross‑linked polyphenylene sulfide)
    Mfg status: manufacturable
  Saved to: /orcd/pool/004/tphage/SG_march/MARS_ablat

## Summary

In [9]:
print(f"{'Query':<20} {'Candidate':<50} {'Status':<15} {'Time (s)':<10}")
print("-" * 95)
for name, result in results.items():
    cand = result["candidate_selection"]["final_candidate"]
    cand_name = (cand.get("material_name", "N/A") if cand else "N/A")[:48]
    status = result["manufacturing_process"]["status"]
    dur = result["metadata"]["duration_seconds"]
    print(f"{name:<20} {cand_name:<50} {status:<15} {dur:<10.1f}")

Query                Candidate                                          Status          Time (s)  
-----------------------------------------------------------------------------------------------
Query1               DIC.PPS (cross‑linked polyphenylene sulfide)       manufacturable  243.7     
Query2               Polyethylene Naphthalate (PEN) Heat Shrink Tubin   manufacturable  135.3     
Query3               AKROTEK® PEEK 8 natural                            manufacturable  87.9      
Query4               Polydimethylsiloxane (PDMS) – 3 wt% additive in    manufacturable  97.3      
Query5               DIC.PPS                                            manufacturable  100.3     
PTFE_Seals           PEEK 450G (VICTREX™)                               manufacturable  90.2      
